# PTD Scraper - Planos de Transformação Digital (Gov.br)

Notebook para coleta, extração e análise dos Planos de Transformação Digital dos órgãos federais brasileiros.

**Fonte:** [gov.br/governodigital - Planos de Transformação Digital](https://www.gov.br/governodigital/pt-br/estrategias-e-governanca-digital/planos-de-transformacao-digital)

**Pipeline:**
1. Scraping da lista de órgãos signatários e URLs dos PDFs
2. Download dos PDFs (Documento Diretivo + Anexo de Entregas)
3. Extração de tabelas com Docling (IBM)
4. Padronização de vocabulário
5. Exportação CSV/JSON + relatório de erros
6. Análises estatísticas do corpus

In [ ]:
# ============================================================
# CÉLULA 1 — Setup & Instalação
# ============================================================
# Monta Google Drive e instala dependências.
# Em Colab, execute esta célula primeiro.

import os, sys

# --- Google Drive (descomente em Colab) ---
# from google.colab import drive
# drive.mount('/content/drive')

# --- Instalação de pacotes ---
# !pip install -q "typer>=0.24.0" "typer-slim>=0.24.0"
# !pip install -q docling beautifulsoup4 requests tqdm pandas matplotlib seaborn tabulate

# --- Diretórios de trabalho ---
# Em Colab aponte para o Drive; localmente use ./output
USE_DRIVE = False
if USE_DRIVE:
    BASE_DIR = "/content/drive/MyDrive/PTD_Scraper"
else:
    BASE_DIR = os.path.join(os.getcwd(), "ptd_output")

DIRS = {
    "pdfs_diretivo":  os.path.join(BASE_DIR, "pdfs", "diretivo"),
    "pdfs_entregas":  os.path.join(BASE_DIR, "pdfs", "entregas"),
    "output":         os.path.join(BASE_DIR, "output"),
    "checkpoints":    os.path.join(BASE_DIR, "checkpoints"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print(f"Diretório base: {BASE_DIR}")
print("Estrutura criada:", list(DIRS.keys()))

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração, Constantes e Estruturas de Dados
# ============================================================
import time, pickle, unicodedata, re, json, logging, difflib
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Tuple, Dict, Any
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# --------------- URLs e parâmetros de rede ------------------
BASE_URL = ("https://www.gov.br/governodigital/pt-br/"
            "estrategias-e-governanca-digital/"
            "planos-de-transformacao-digital")
REQUEST_DELAY = 2.0        # segundos entre requests ao gov.br
MAX_RETRIES   = 4
REQUEST_TIMEOUT = 90
HTTP_HEADERS  = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0 Safari/537.36"),
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.5",
}

# --------------- Vocabulários canônicos ---------------------
CANONICAL_EIXOS = [
    "Serviços Digitais e Melhoria da Qualidade",
    "Unificação de Canais Digitais",
    "Governança e Gestão de Dados",
    "Segurança e Privacidade",
    "Projetos Especiais",
]

# Mapa Eixo → lista de Produtos (44 produtos oficiais)
CANONICAL_PRODUTOS: Dict[str, List[str]] = {
    "Serviços Digitais e Melhoria da Qualidade": [
        "Disponibilização em Acesso Digital",
        "Disponibilização de datas/cronograma na Agenda Gov.Br",
        "Evolução do Serviço",
        "Implantação da Área Logada Gov.Br",
        "Implantação da Experiência LabQ",
        "Implementação do VLIBRAS",
        "Integração à ferramenta de acompanhamento das solicitações",
        "Integração à ferramenta de avaliação da satisfação dos usuários",
        "Realização de Autodiagnóstico de Qualidade",
        "Revisão da descrição dos serviços",
        "Oferta de agendamento digital",
        "Integração à ferramenta de solicitação de atendimento presencial",
        "Implantação do Atendimento Virtual",
    ],
    "Unificação de Canais Digitais": [
        "Implantação do Design System",
        "Integração ao Login Único",
        "Integração ao Pagtesouro",
        "Integração com Assinatura Digital Gov.br",
        "Migração de APPs móveis para a loja do Gov.br",
        "Migração de Portal Institucional",
        "Migração de Serviço para Plataforma Unificada",
    ],
    "Governança e Gestão de Dados": [
        "Abertura de bases de dados",
        "Adequação à LGPD",
        "Catalogação de bases de dados no Portal de Dados",
        "Criação de Comitê de Governança de Dados",
        "Elaboração de Plano de Dados Abertos",
        "Elaboração do PDTIC",
        "Elaboração/Revisão da POSIC",
        "Implantação de processo de gestão de dados",
        "Implantação do inventário de bases de dados",
        "Implementação da interoperabilidade de dados",
        "Integração de dados ao Conecta Gov.br",
        "Integração de dados ao Datamart",
        "Melhoria da qualidade de bases de dados",
        "Nomeação de Encarregado de Dados Pessoais",
        "Obtenção de certificação de bases de dados",
        "Publicação de dados no Portal de Dados Abertos",
        "Realização de Inventário de Dados",
        "Relatório de Impacto à Proteção de Dados Pessoais",
        "Resposta a demandas de compartilhamento",
        "Utilização de dados do Conecta Gov.br",
    ],
    "Segurança e Privacidade": [
        "Adequação ao Framework de Privacidade e Segurança da Informação",
        "Elevação do nível de maturidade em privacidade e segurança",
    ],
    "Projetos Especiais": [
        "Iniciativa de Transformação Digital",
        "Projeto Especial de Transformação Digital",
    ],
}

# Lista flat de todos os produtos para busca
ALL_PRODUTOS = [p for prods in CANONICAL_PRODUTOS.values() for p in prods]

# Mapa reverso: produto → eixo canônico
PRODUTO_TO_EIXO = {}
for eixo, prods in CANONICAL_PRODUTOS.items():
    for p in prods:
        PRODUTO_TO_EIXO[p] = eixo

# Escalas do Documento Diretivo (tabela de riscos)
PROBABILIDADE_SCALE = [
    "raro", "pouco provável", "provável",
    "muito provável", "praticamente certo",
]
IMPACTO_SCALE = [
    "muito baixo", "baixo", "médio", "alto", "muito alto",
]
TRATAMENTO_OPTIONS = ["mitigar", "eliminar", "transferir", "aceitar"]

# ---------- Órgãos que compartilham PDFs (grupos) ----------
ORGAN_GROUPS: Dict[str, List[str]] = {
    "MD":   ["MD", "CEX", "CM", "COMAER", "CENSIPAM", "FOSORIO", "HFA"],
    "MEC":  ["MEC", "CAPES", "EBSERH", "FNDE", "FUNDAJ", "IBC", "INEP", "INES"],
    "MF":   ["MF", "RFB", "STN", "PGFN"],
    "MMA":  ["MMA", "IBAMA", "ICMBIO", "SFB", "JBRJ"],
    "MT":   ["MT", "ANTT", "DNIT"],
    "MIDR": ["MIDR", "CODEVASF", "SUDAM", "SUDECO", "SUDENE"],
    "MDA":  ["MDA", "CONAB", "INCRA"],
}
# Reverso: sigla membro → sigla cabeça do grupo
MEMBER_TO_GROUP = {}
for head, members in ORGAN_GROUPS.items():
    for m in members:
        if m != head:
            MEMBER_TO_GROUP[m] = head

# --------------- Dataclasses --------------------------------
@dataclass
class OrganInfo:
    sigla: str
    nome_completo: str
    grupo: Optional[str] = None          # sigla do cabeça, se agrupado
    url_diretivo: Optional[str] = None
    url_entregas: Optional[str] = None
    pdf_path_diretivo: Optional[str] = None
    pdf_path_entregas: Optional[str] = None

@dataclass
class RiskEntry:
    orgao_sigla: str
    risco_texto: str = ""
    probabilidade_original: str = ""
    probabilidade_normalizada: str = ""
    impacto_original: str = ""
    impacto_normalizado: str = ""
    tratamento_original: str = ""
    tratamento_normalizado: str = ""
    acoes_tratamento: str = ""
    extraction_confidence: str = "high"    # high / medium / low
    needs_review: bool = False
    review_reason: Optional[str] = None

@dataclass
class DeliveryEntry:
    orgao_sigla: str
    tabela_tipo: str = ""                  # pactuada / concluida / cancelada
    servico_acao: str = ""
    produto_original: str = ""
    produto_normalizado: str = ""
    eixo_original: str = ""
    eixo_normalizado: str = ""
    area_responsavel: Optional[str] = None
    data_pactuada: Optional[str] = None
    data_entrega: Optional[str] = None
    pactuado: Optional[str] = None         # Sim / Não (concluídas)
    justificativa: Optional[str] = None    # (canceladas)
    extraction_confidence: str = "high"
    needs_review: bool = False
    review_reason: Optional[str] = None

@dataclass
class ProcessingError:
    orgao_sigla: str
    document_type: str     # diretivo / entregas
    stage: str             # download / extraction / standardization
    error_type: str
    error_message: str
    timestamp: str = ""
    url: Optional[str] = None

    def __post_init__(self):
        if not self.timestamp:
            self.timestamp = datetime.now().isoformat()

# Contêineres globais
all_organs: List[OrganInfo] = []
all_risks: List[RiskEntry] = []
all_deliveries: List[DeliveryEntry] = []
all_errors: List[ProcessingError] = []

print(f"Configuração carregada: {len(ALL_PRODUTOS)} produtos canônicos em {len(CANONICAL_EIXOS)} eixos")

In [ ]:
# ============================================================
# CÉLULA 3 — Funções Utilitárias
# ============================================================

# --------------- Rede com retry -----------------------------
def safe_request(url: str, max_retries: int = MAX_RETRIES,
                 delay: float = REQUEST_DELAY,
                 timeout: int = REQUEST_TIMEOUT) -> Optional[requests.Response]:
    """GET com retry exponencial e rate-limiting."""
    for attempt in range(1, max_retries + 1):
        try:
            time.sleep(delay)
            resp = requests.get(url, headers=HTTP_HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return resp
            if resp.status_code == 503 and attempt < max_retries:
                wait = delay * (2 ** attempt)
                print(f"  503 em {url} — retry {attempt}/{max_retries} em {wait:.0f}s")
                time.sleep(wait)
                continue
            resp.raise_for_status()
        except requests.RequestException as exc:
            if attempt < max_retries:
                wait = delay * (2 ** attempt)
                print(f"  Erro ({exc}) — retry {attempt}/{max_retries} em {wait:.0f}s")
                time.sleep(wait)
            else:
                print(f"  FALHA definitiva: {url} — {exc}")
                return None
    return None

# --------------- Normalização de texto ----------------------
def normalize_text(text: str) -> str:
    """Normaliza unicode, whitespace e caixa para comparação."""
    if not text:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def strip_accents(text: str) -> str:
    """Remove acentos para matching fuzzy."""
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

# --------------- Matching fuzzy de vocabulário --------------
def fuzzy_match(original: str, candidates: list,
                threshold: float = 0.85) -> Tuple[str, float]:
    """Retorna (melhor_candidato, score). Score 0 se abaixo do threshold."""
    if not original or not candidates:
        return ("", 0.0)
    norm = normalize_text(original).lower()
    norm_stripped = strip_accents(norm)
    # Tentativa exata (case-insensitive, accent-insensitive)
    for c in candidates:
        c_norm = normalize_text(c).lower()
        if norm == c_norm or norm_stripped == strip_accents(c_norm):
            return (c, 1.0)
    # Fuzzy
    best, best_score = "", 0.0
    for c in candidates:
        c_norm = normalize_text(c).lower()
        score = difflib.SequenceMatcher(None, norm_stripped,
                                        strip_accents(c_norm)).ratio()
        if score > best_score:
            best, best_score = c, score
    if best_score >= threshold:
        return (best, best_score)
    return (best, best_score)   # retorna melhor mesmo abaixo do threshold

def fuzzy_match_produto(original: str) -> Tuple[str, float]:
    return fuzzy_match(original, ALL_PRODUTOS, threshold=0.80)

def fuzzy_match_eixo(original: str) -> Tuple[str, float]:
    return fuzzy_match(original, CANONICAL_EIXOS, threshold=0.80)

def fuzzy_match_scale(original: str, scale: list) -> Tuple[str, float]:
    return fuzzy_match(original, scale, threshold=0.70)

# --------------- Parse de datas variadas --------------------
_DATE_PATTERNS = [
    (r"(\d{2})/(\d{2})/(\d{4})", lambda m: f"{m.group(3)}-{m.group(2)}-{m.group(1)}"),
    (r"(\d{2})/(\d{4})",          lambda m: f"{m.group(2)}-{m.group(1)}"),
    (r"(\d{4})-(\d{2})-(\d{2})",  lambda m: m.group(0)),
]
_MONTH_MAP = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04",
    "mai": "05", "jun": "06", "jul": "07", "ago": "08",
    "set": "09", "out": "10", "nov": "11", "dez": "12",
}

def parse_date(text: str) -> Optional[str]:
    """Converte formatos variados para YYYY-MM ou YYYY-MM-DD."""
    if not text:
        return None
    text = normalize_text(text).strip()
    for pat, fmt in _DATE_PATTERNS:
        m = re.search(pat, text)
        if m:
            return fmt(m)
    # Formato "jun/25", "mar/2026"
    m = re.match(r"([a-záéíóú]{3})[\./\-](\d{2,4})", text.lower())
    if m:
        month = _MONTH_MAP.get(m.group(1)[:3])
        year = m.group(2)
        if len(year) == 2:
            year = "20" + year
        if month:
            return f"{year}-{month}"
    return text   # retorna original se não parsear

# --------------- Checkpoint / Resume ------------------------
def save_checkpoint(data: Any, name: str) -> None:
    path = os.path.join(DIRS["checkpoints"], f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint salvo: {name}")

def load_checkpoint(name: str) -> Optional[Any]:
    path = os.path.join(DIRS["checkpoints"], f"{name}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint carregado: {name}")
        return data
    return None

# --------------- Logging ------------------------------------
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("ptd_scraper")

print("Funções utilitárias carregadas.")

## 2. Scraping — Lista de Órgãos Signatários

Extrai os 91 órgãos e seus links de PDFs (Documento Diretivo + Anexo de Entregas) da página principal.

In [ ]:
# ============================================================
# CÉLULA 4 — Scraping: Lista de Órgãos e URLs dos PDFs
# ============================================================

def _classify_pdf_link(anchor_text: str) -> Optional[str]:
    """Classifica um link de PDF como 'diretivo' ou 'entregas' pelo texto da âncora."""
    text = normalize_text(anchor_text).lower()
    text_no_accents = strip_accents(text)

    # Documento Diretivo
    diretivo_kw = ["diretivo", "documento diretivo"]
    if any(kw in text_no_accents for kw in diretivo_kw):
        return "diretivo"

    # Anexo de Entregas
    entregas_kw = ["entregas", "anexo de entregas", "anexo entregas"]
    if any(kw in text_no_accents for kw in entregas_kw):
        return "entregas"

    return None


def _parse_sigla_nome(raw_text: str) -> List[Tuple[str, str]]:
    """
    Extrai pares (sigla, nome_completo) de um texto de cabeçalho de órgão.

    Formatos comuns:
      "SIGLA - Nome Completo"
      "SIGLA1 / SIGLA2 - Nome Completo"   (órgãos agrupados)
      "SIGLA – Nome Completo"              (meia-risca)
    """
    text = normalize_text(raw_text)
    if not text:
        return []

    # Separa sigla(s) do nome pelo primeiro traço (- ou –)
    match = re.match(r"^(.+?)\s*[-–]\s*(.+)$", text)
    if not match:
        return []

    sigla_part = match.group(1).strip()
    nome_part = match.group(2).strip()

    # Pode haver múltiplas siglas: "SIGLA1 / SIGLA2"
    siglas = [s.strip() for s in re.split(r"\s*/\s*", sigla_part) if s.strip()]
    # Validação: siglas são uppercase e curtas
    siglas = [s for s in siglas if re.match(r"^[A-ZÁÉÍÓÚÂÊÔÃÕÇ]{2,12}$", s)]

    if not siglas:
        return []

    return [(s, nome_part) for s in siglas]


def scrape_organ_listing(url: str) -> List[OrganInfo]:
    """
    Faz o scraping da página principal do gov.br para extrair a lista
    de órgãos signatários e seus links de PDFs.

    Estratégia:
      1. Encontra todos os links que apontam para PDFs em 'ptds-vigentes/'
      2. Para cada link, localiza o cabeçalho do órgão mais próximo acima
      3. Classifica o link como 'diretivo' ou 'entregas' pelo texto da âncora
      4. Agrupa por sigla, expandindo órgãos que compartilham PDFs
    """
    resp = safe_request(url)
    if resp is None:
        raise RuntimeError(f"Não foi possível acessar {url}")

    soup = BeautifulSoup(resp.content, "html.parser")

    # Encontra a área de conteúdo principal
    content = (
        soup.find("article")
        or soup.find("div", {"id": "content-core"})
        or soup.find("div", class_=re.compile(r"(content|texto|article)", re.I))
        or soup.body
    )
    if content is None:
        raise RuntimeError("Não encontrou área de conteúdo na página")

    # ---- Fase 1: coletar todos os cabeçalhos de órgãos ----
    # Procuramos <strong> ou <b> que contenham padrão "SIGLA - Nome"
    organ_headers = []   # [(element, [(sigla, nome)])]
    for tag in content.find_all(["strong", "b"]):
        raw = tag.get_text(separator=" ", strip=True)
        parsed = _parse_sigla_nome(raw)
        if parsed:
            organ_headers.append((tag, parsed))

    # ---- Fase 2: coletar todos os links de PDF ----
    pdf_links = []   # [(element, href, doc_type)]
    for a_tag in content.find_all("a", href=True):
        href = a_tag["href"]
        if "ptds-vigentes/" not in href:
            continue
        if not href.lower().endswith(".pdf"):
            continue
        anchor_text = a_tag.get_text(separator=" ", strip=True)
        doc_type = _classify_pdf_link(anchor_text)
        if doc_type is None:
            # Tentativa pelo nome do arquivo
            fname_lower = href.rsplit("/", 1)[-1].lower()
            if "diretivo" in fname_lower:
                doc_type = "diretivo"
            elif "entregas" in fname_lower or "anexo" in fname_lower:
                doc_type = "entregas"
            else:
                # Default heuristic: primeiro link = diretivo, segundo = entregas
                doc_type = "unknown"
        # Converte URL relativa para absoluta
        if href.startswith("/"):
            href = "https://www.gov.br" + href
        pdf_links.append((a_tag, href, doc_type))

    logger.info(f"Encontrados {len(organ_headers)} cabeçalhos de órgãos e {len(pdf_links)} links de PDF")

    # ---- Fase 3: associar cada PDF ao órgão mais próximo acima ----
    # Obtemos posição de cada elemento na árvore linearizada
    all_elements = list(content.descendants)

    def _element_index(el):
        """Retorna o índice do elemento na lista linearizada de descendentes."""
        try:
            # Percorre para cima até encontrar o próprio elemento ou um ancestral direto
            for idx, node in enumerate(all_elements):
                if node is el:
                    return idx
        except Exception:
            pass
        return -1

    # Cache de posições
    header_positions = []
    for tag, parsed in organ_headers:
        idx = _element_index(tag)
        header_positions.append((idx, parsed))

    # Ordena por posição
    header_positions.sort(key=lambda x: x[0])

    # Para cada link PDF, encontra o cabeçalho imediatamente anterior
    organ_data: Dict[str, Dict[str, Optional[str]]] = {}
    # sigla → {"nome": ..., "url_diretivo": ..., "url_entregas": ...}

    for a_tag, href, doc_type in pdf_links:
        link_idx = _element_index(a_tag)

        # Encontra o cabeçalho mais próximo antes deste link
        best_header = None
        for h_idx, parsed in header_positions:
            if h_idx <= link_idx:
                best_header = parsed
            else:
                break

        if best_header is None:
            logger.warning(f"PDF sem órgão associado: {href}")
            continue

        # Registra para todas as siglas desse cabeçalho
        for sigla, nome in best_header:
            if sigla not in organ_data:
                organ_data[sigla] = {"nome": nome, "url_diretivo": None, "url_entregas": None}

            if doc_type == "diretivo":
                organ_data[sigla]["url_diretivo"] = href
            elif doc_type == "entregas":
                organ_data[sigla]["url_entregas"] = href
            elif doc_type == "unknown":
                # Primeiro desconhecido vai para diretivo, segundo para entregas
                if organ_data[sigla]["url_diretivo"] is None:
                    organ_data[sigla]["url_diretivo"] = href
                elif organ_data[sigla]["url_entregas"] is None:
                    organ_data[sigla]["url_entregas"] = href

    # ---- Fase 4: expandir órgãos agrupados ----
    # Órgãos membros de um grupo herdam os PDFs da sigla-cabeça
    expanded: Dict[str, Dict[str, Optional[str]]] = dict(organ_data)

    for head_sigla, members in ORGAN_GROUPS.items():
        if head_sigla in organ_data:
            head_info = organ_data[head_sigla]
            for member in members:
                if member == head_sigla:
                    continue
                if member not in expanded:
                    expanded[member] = {
                        "nome": head_info["nome"],
                        "url_diretivo": head_info["url_diretivo"],
                        "url_entregas": head_info["url_entregas"],
                    }
                else:
                    # Preenche URLs ausentes com as do cabeça do grupo
                    if expanded[member]["url_diretivo"] is None:
                        expanded[member]["url_diretivo"] = head_info["url_diretivo"]
                    if expanded[member]["url_entregas"] is None:
                        expanded[member]["url_entregas"] = head_info["url_entregas"]

    # ---- Fase 5: construir lista de OrganInfo ----
    organs: List[OrganInfo] = []
    for sigla in sorted(expanded.keys()):
        info = expanded[sigla]
        grupo = MEMBER_TO_GROUP.get(sigla)
        organs.append(OrganInfo(
            sigla=sigla,
            nome_completo=info["nome"],
            grupo=grupo,
            url_diretivo=info.get("url_diretivo"),
            url_entregas=info.get("url_entregas"),
        ))

    return organs


# ---- Execução ----
_cached = load_checkpoint("organ_listing")
if _cached is not None:
    all_organs = _cached
    print(f"Carregado do checkpoint: {len(all_organs)} órgãos")
else:
    print("Fazendo scraping da página principal...")
    all_organs = scrape_organ_listing(BASE_URL)
    save_checkpoint(all_organs, "organ_listing")

# ---- Validação e Resumo ----
_n_total = len(all_organs)
_n_diretivo = sum(1 for o in all_organs if o.url_diretivo)
_n_entregas = sum(1 for o in all_organs if o.url_entregas)
_n_ambos = sum(1 for o in all_organs if o.url_diretivo and o.url_entregas)
_n_nenhum = sum(1 for o in all_organs if not o.url_diretivo and not o.url_entregas)
_n_grupos = sum(1 for o in all_organs if o.grupo is not None)

print(f"\n{'='*50}")
print(f"Total de órgãos encontrados: {_n_total}")
if _n_total < 80 or _n_total > 110:
    print(f"  ATENÇÃO: esperados ~91 órgãos, encontrados {_n_total}")
else:
    print(f"  Contagem dentro do esperado (~91)")
print(f"  Com Documento Diretivo:    {_n_diretivo}")
print(f"  Com Anexo de Entregas:     {_n_entregas}")
print(f"  Com ambos:                 {_n_ambos}")
print(f"  Sem nenhum PDF:            {_n_nenhum}")
print(f"  Membros de grupo:          {_n_grupos}")
print(f"{'='*50}")

# Lista órgãos sem PDFs para revisão
if _n_nenhum > 0:
    print("\nÓrgãos SEM nenhum PDF:")
    for o in all_organs:
        if not o.url_diretivo and not o.url_entregas:
            print(f"  - {o.sigla} ({o.nome_completo})")

# Amostra
print("\nAmostra (primeiros 5):")
for o in all_organs[:5]:
    print(f"  {o.sigla:12s} | dir={'Sim' if o.url_diretivo else '---'} "
          f"| ent={'Sim' if o.url_entregas else '---'} "
          f"| grupo={o.grupo or '—'}")

## 3. Download dos PDFs

Baixa todos os PDFs identificados com rate-limiting e verificação de integridade.

In [ ]:
# ============================================================
# CÉLULA 5 — Download dos PDFs
# ============================================================

PDF_MAGIC_BYTES = b"%PDF"
MIN_SUSPICIOUS_SIZE = 10 * 1024   # 10 KB
MIN_VALID_SIZE = 1000             # 1 KB (skip threshold para resume)


def _download_single_pdf(url: str, dest_path: str, sigla: str,
                         doc_type: str) -> Optional[ProcessingError]:
    """
    Baixa um único PDF. Retorna ProcessingError em caso de falha, None se OK.
    Pula o download se o arquivo já existe com tamanho > MIN_VALID_SIZE.
    """
    # Resume: pula se já existe e parece válido
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > MIN_VALID_SIZE:
        return None

    resp = safe_request(url)
    if resp is None:
        return ProcessingError(
            orgao_sigla=sigla,
            document_type=doc_type,
            stage="download",
            error_type="request_failed",
            error_message=f"Falha ao baixar após {MAX_RETRIES} tentativas",
            url=url,
        )

    content = resp.content

    # Verifica magic bytes
    if not content[:4].startswith(PDF_MAGIC_BYTES):
        return ProcessingError(
            orgao_sigla=sigla,
            document_type=doc_type,
            stage="download",
            error_type="invalid_pdf",
            error_message=f"Arquivo não começa com %PDF (magic bytes: {content[:8]!r})",
            url=url,
        )

    # Salva o arquivo
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    with open(dest_path, "wb") as f:
        f.write(content)

    # Alerta para arquivos suspeitamente pequenos
    file_size = len(content)
    if file_size < MIN_SUSPICIOUS_SIZE:
        logger.warning(f"  {sigla}/{doc_type}: arquivo pequeno ({file_size:,} bytes) — pode estar corrompido")

    return None


def download_all_pdfs(organs: List[OrganInfo]) -> List[ProcessingError]:
    """
    Baixa todos os PDFs (diretivo + entregas) dos órgãos.
    Atualiza organ.pdf_path_diretivo e organ.pdf_path_entregas.
    Retorna lista de erros.
    """
    errors: List[ProcessingError] = []
    downloaded_urls: Dict[str, str] = {}   # url → caminho local (evita re-download)

    # Contagem total de downloads potenciais
    total_downloads = sum(
        (1 if o.url_diretivo else 0) + (1 if o.url_entregas else 0)
        for o in organs
    )

    pbar = tqdm(total=total_downloads, desc="Baixando PDFs", unit="pdf")

    for organ in organs:
        for doc_type, url_attr, path_attr, target_dir in [
            ("diretivo", "url_diretivo", "pdf_path_diretivo", DIRS["pdfs_diretivo"]),
            ("entregas", "url_entregas", "pdf_path_entregas", DIRS["pdfs_entregas"]),
        ]:
            url = getattr(organ, url_attr)
            if url is None:
                continue

            filename = f"{organ.sigla}_{doc_type}.pdf"
            dest_path = os.path.join(target_dir, filename)

            # Se outro órgão do mesmo grupo já baixou este URL, reutiliza
            if url in downloaded_urls:
                existing_path = downloaded_urls[url]
                if os.path.exists(existing_path):
                    # Copia ou cria link — usamos cópia para robustez
                    if not os.path.exists(dest_path):
                        import shutil
                        shutil.copy2(existing_path, dest_path)
                    setattr(organ, path_attr, dest_path)
                    pbar.update(1)
                    continue

            err = _download_single_pdf(url, dest_path, organ.sigla, doc_type)
            if err is not None:
                errors.append(err)
            else:
                setattr(organ, path_attr, dest_path)
                downloaded_urls[url] = dest_path

            pbar.update(1)

    pbar.close()
    return errors


# ---- Execução ----
_ckpt_downloads = load_checkpoint("downloads_done")
if _ckpt_downloads is not None:
    # Restaura caminhos locais a partir do checkpoint
    _path_map = _ckpt_downloads  # Dict[sigla, {"diretivo": path, "entregas": path}]
    _restored = 0
    for organ in all_organs:
        if organ.sigla in _path_map:
            paths = _path_map[organ.sigla]
            if paths.get("diretivo") and os.path.exists(paths["diretivo"]):
                organ.pdf_path_diretivo = paths["diretivo"]
                _restored += 1
            if paths.get("entregas") and os.path.exists(paths["entregas"]):
                organ.pdf_path_entregas = paths["entregas"]
                _restored += 1
    print(f"Checkpoint restaurado: {_restored} caminhos de PDF recarregados")
    download_errors = []
else:
    print("Iniciando download dos PDFs...")
    download_errors = download_all_pdfs(all_organs)
    all_errors.extend(download_errors)

    # Salva checkpoint com mapa de caminhos
    _path_map = {}
    for organ in all_organs:
        _path_map[organ.sigla] = {
            "diretivo": organ.pdf_path_diretivo,
            "entregas": organ.pdf_path_entregas,
        }
    save_checkpoint(_path_map, "downloads_done")
    save_checkpoint(all_organs, "organ_listing")   # atualiza com caminhos

# ---- Resumo ----
_n_dir_ok = sum(1 for o in all_organs if o.pdf_path_diretivo and os.path.exists(o.pdf_path_diretivo))
_n_ent_ok = sum(1 for o in all_organs if o.pdf_path_entregas and os.path.exists(o.pdf_path_entregas))

_total_size = 0
_suspicious: List[str] = []
for o in all_organs:
    for p in [o.pdf_path_diretivo, o.pdf_path_entregas]:
        if p and os.path.exists(p):
            sz = os.path.getsize(p)
            _total_size += sz
            if sz < MIN_SUSPICIOUS_SIZE:
                _suspicious.append(f"{o.sigla}: {os.path.basename(p)} ({sz:,} bytes)")

print(f"\n{'='*50}")
print(f"Download concluído")
print(f"  Documento Diretivo OK:    {_n_dir_ok}")
print(f"  Anexo de Entregas OK:     {_n_ent_ok}")
print(f"  Erros de download:        {len(download_errors)}")
print(f"  Tamanho total:            {_total_size / (1024*1024):.1f} MB")
print(f"{'='*50}")

if _suspicious:
    print(f"\nArquivos suspeitamente pequenos (<{MIN_SUSPICIOUS_SIZE//1024} KB):")
    for s in _suspicious:
        print(f"  - {s}")

if download_errors:
    print(f"\nErros de download:")
    for e in download_errors:
        print(f"  - {e.orgao_sigla}/{e.document_type}: {e.error_type} — {e.error_message}")

## 4. Configuração do Docling e Classificação de Tabelas

Inicializa o pipeline de extração de tabelas Docling (IBM) e define funções de classificação.

In [ ]:
# ============================================================
# CÉLULA 6 — Configuração do Docling e Classificação de Tabelas
# ============================================================

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.base_models import InputFormat

# --------------- Criação do conversor -------------------------

def create_converter(accurate: bool = True, ocr: bool = True) -> DocumentConverter:
    """Cria um DocumentConverter configurado para extração de tabelas.

    Args:
        accurate: Se True usa TableFormerMode.ACCURATE; senão FAST.
        ocr: Se True habilita OCR para PDFs escaneados.

    Returns:
        DocumentConverter pronto para uso.
    """
    pipeline_opts = PdfPipelineOptions()
    pipeline_opts.do_table_structure = True
    pipeline_opts.table_structure_options.mode = (
        TableFormerMode.ACCURATE if accurate else TableFormerMode.FAST
    )

    # OCR
    pipeline_opts.do_ocr = ocr

    # Desabilita features desnecessárias
    pipeline_opts.generate_picture_images = False
    pipeline_opts.images_scale = 1.0

    format_options = {
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_opts),
    }

    return DocumentConverter(format_options=format_options)


# --------------- Classificação de tabelas diretivo ------------

def _normalize_header(text: str) -> str:
    """Normaliza texto de cabeçalho para comparação: minúscula, sem acentos, sem espaços extras."""
    if not isinstance(text, str):
        return ""
    t = normalize_text(text).lower()
    return strip_accents(t)


def classify_diretivo_table(df: pd.DataFrame) -> str:
    """Classifica uma tabela extraída de Documento Diretivo.

    Returns:
        'risk_table' — tabela de riscos e tratamento
        'organ_info' — informações institucionais
        'signature'  — bloco de assinaturas
        'unknown'    — não classificada
    """
    if df is None or df.empty:
        return "unknown"

    ncols = len(df.columns)
    headers = [_normalize_header(str(c)) for c in df.columns]
    # Também verificar a primeira linha caso os headers sejam genéricos (0,1,2...)
    first_row_headers = []
    if len(df) > 0:
        first_row_headers = [_normalize_header(str(v)) for v in df.iloc[0]]

    all_headers = headers + first_row_headers

    combined = " ".join(all_headers)

    # --- Tabela de riscos: espera 4-6 colunas com keywords específicas ---
    risk_keywords = ["risco", "probabilidade", "impacto", "tratamento"]
    risk_alt_keywords = ["evento", "classificacao", "severidade", "resposta", "acao", "acoes"]
    risk_hits = sum(1 for kw in risk_keywords if kw in combined)
    risk_alt_hits = sum(1 for kw in risk_alt_keywords if kw in combined)

    if risk_hits >= 2 and 3 <= ncols <= 8:
        return "risk_table"
    if risk_hits >= 1 and risk_alt_hits >= 1 and 3 <= ncols <= 8:
        return "risk_table"

    # --- Informações do órgão: keywords institucionais ---
    info_keywords = [
        "orgao", "ministerio", "secretaria", "sigla", "cnpj",
        "responsavel", "gestor", "dirigente", "titular", "autoridade",
        "instituicao", "vinculacao",
    ]
    info_hits = sum(1 for kw in info_keywords if kw in combined)
    if info_hits >= 2:
        return "organ_info"

    # --- Bloco de assinatura ---
    sig_keywords = ["assinatura", "assinado", "data", "nome", "cargo", "cpf"]
    sig_hits = sum(1 for kw in sig_keywords if kw in combined)
    if sig_hits >= 2 and ncols <= 4:
        return "signature"

    return "unknown"


# --------------- Classificação de tabelas entregas ------------

def classify_entregas_table(df: pd.DataFrame) -> str:
    """Classifica uma tabela extraída de Anexo de Entregas.

    Returns:
        'pactuadas'   — entregas pactuadas (agendadas)
        'concluidas'  — entregas concluídas
        'canceladas'  — entregas canceladas
        'unknown'     — não classificada
    """
    if df is None or df.empty:
        return "unknown"

    headers = [_normalize_header(str(c)) for c in df.columns]
    first_row_headers = []
    if len(df) > 0:
        first_row_headers = [_normalize_header(str(v)) for v in df.iloc[0]]

    all_headers = headers + first_row_headers
    combined = " ".join(all_headers)

    has_area_resp = any(
        "area" in h and "responsavel" in h for h in all_headers
    ) or "area responsavel" in combined or "arearesponsavel" in combined.replace(" ", "")

    has_dt_pactuada = (
        "dtpactuada" in combined.replace(" ", "")
        or "data pactuada" in combined
        or "dt pactuada" in combined
        or "datapactuada" in combined.replace(" ", "")
    )

    has_dt_entrega = (
        "dtentrega" in combined.replace(" ", "")
        or "data entrega" in combined
        or "dt entrega" in combined
        or "dataentrega" in combined.replace(" ", "")
        or "data de entrega" in combined
    )

    has_pactuado_flag = (
        "pactuado?" in combined
        or "pactuado ?" in combined
        or ("pactuado" in combined and ("sim" in combined or "nao" in combined))
    )

    has_justificativa = (
        "justificativa" in combined
        or "motivo" in combined and "cancelamento" in combined
    )

    # Classificação por prioridade (canceladas são mais restritivas)
    # Canceladas: tem Justificativa
    if has_justificativa:
        return "canceladas"

    # Concluídas: tem DtEntrega ou coluna "Pactuado?"
    if has_dt_entrega or has_pactuado_flag:
        return "concluidas"

    # Pactuadas: tem Area Responsável E DtPactuada
    if has_area_resp and has_dt_pactuada:
        return "pactuadas"

    # Fallback: se tem DtPactuada mas sem Area (layout alternativo)
    if has_dt_pactuada:
        return "pactuadas"

    return "unknown"


# --------------- Instância default ----------------------------

converter_accurate = create_converter(accurate=True, ocr=True)

print("Docling configurado (modo ACCURATE + OCR).")
print("Classificadores de tabelas carregados.")

## 5. Extração de Tabelas de Risco (Documentos Diretivos)

Extrai e normaliza a tabela de riscos de cada Documento Diretivo usando Docling.

## 6. Extração de Tabelas de Entregas (Anexos de Entregas)

Extrai tabelas de entregas pactuadas, concluídas e canceladas de cada Anexo de Entregas.

## 7. Padronização de Vocabulário

Normaliza produtos, eixos, probabilidades, impactos e tratamentos usando matching em camadas.

## 8. Exportação de Dados

Exporta dados processados em CSV e JSON, mais relatório de erros e ajustes.

## 9. Análises Estatísticas e Visualizações

Estatísticas descritivas do corpus e visualizações dos dados extraídos.

## 10. Suporte a Iterações e Correções Manuais

Ferramentas para revisão manual e aplicação de correções no próximo ciclo.